# Environment & Git Setup Notes

Personal reference for how this project's conda environment and GitHub repo were set up, plus the activation fix needed for PowerShell. Commands are PowerShell / Git commands, not Python — copy-paste into a terminal, don't run these cells.

## 1. Create the environment in VS Code

Despite the folder being named `venv`, this was created as a **conda environment** (via `-p`, i.e. `--prefix`), not a standard Python `venv`. That distinction matters later for activation.

```powershell
conda create -p venv python==3.8 -y
conda activate venv/
```

## 2. Initialize git and make the first commit

```powershell
git init
git add README.md
git commit -m "first commit"
git status
```

## 3. Rename branch and connect to GitHub

```powershell
git branch -M main
git remote add origin https://github.com/Satya666-pra/MLPROJECT.git
git remote -v
git config --global user.email
git push -u origin main
```

## 4. Sync remote changes

```powershell
git pull
```

## 5. Gotcha: `pip install` fails with "externally-managed-environment"

This happened when `pip install -r requirements.txt` was run without the environment properly activated in that terminal — pip fell back to the protected base/system Python instead of the project's env.

**Root cause found:** `conda activate` alone did nothing in a plain PowerShell session because conda's shell hook wasn't initialized there yet (`conda` resolved to the raw `.exe`, not the shell-integrated function). So `$env:CONDA_PREFIX` never got set, and `python`/`pip` kept pointing outside the env.

**One-time permanent fix:**
```powershell
conda init powershell
```
Then **close and reopen the terminal**. After that, plain `conda activate` works in every new PowerShell window.

**If you ever see the hook missing again** (e.g. a terminal opened in a weird way), load it manually for that one session:
```powershell
(& "C:\Users\satya\miniconda3-pbe\Scripts\conda.exe" "shell.powershell" "hook") | Out-String | Invoke-Expression
conda activate ".\venv"
```

## 6. Correct day-to-day workflow (after the one-time fix above)

```powershell
conda activate ".\venv"
pip install -r requirements.txt
```

**How to verify the env is actually active before installing anything:**
```powershell
$env:CONDA_PREFIX        # should show the path ending in \venv
python --version         # should show Python 3.8.0
pip --version             # path shown should be inside \venv\lib\site-packages\pip
```

## 7. What `setup.py` does and why it exists

`setup.py` is what turns this folder into an installable Python **package** (named `mlproject`) instead of just a loose collection of scripts. It's what runs behind the scenes when `pip install -e .` is executed.

```python
from setuptools import find_packages, setup
from typing import List

HYPEN_E_DOT = '-e .'
def get_requirements(file_path: str) -> List[str]:
    '''
    this function will return the list of requirements
    '''
    requirements = []
    with open(file_path) as file_obj:
        requirements = file_obj.readlines()
        requirements = [req.replace("\n", "") for req in requirements]

        if HYPEN_E_DOT in requirements:
            requirements.remove(HYPEN_E_DOT)

    return requirements

setup(
    name='mlproject',
    version='0.0.1',
    author='SatyaprakashKabi',
    author_email='satyaprakashKabi@gmail.com',
    packages=find_packages(),
    install_requires=get_requirements('requirements.txt')
)
```

**Line-by-line:**

- `from setuptools import find_packages, setup` — `setup()` is the function that actually registers the package's metadata with pip/setuptools. `find_packages()` auto-discovers every directory containing an `__init__.py` (i.e. every importable sub-package) so they don't have to be listed by hand.

- `HYPEN_E_DOT = '-e .'` — a constant holding the literal text `-e .`, the line that appears at the bottom of `requirements.txt`. `-e .` tells pip "install *this* project itself, in editable mode" (see below). It's a pip-only instruction, not a real package name, so it must never be passed to `install_requires`.

- `get_requirements(file_path)` — reads `requirements.txt` and turns it into the list of dependency strings `setup()` needs:
  - `file_obj.readlines()` reads every line, including its trailing `\n`.
  - the list comprehension strips those `\n` characters off each line.
  - the `if HYPEN_E_DOT in requirements` check removes the `-e .` entry if present — **this is the important guard**. Without it, `setup()` would try to treat `-e .` as a package name and installation would break, because `-e .` only makes sense as a pip command-line flag, not as an item in `install_requires`.

- `setup(...)` — the actual package registration:
  - `name='mlproject'` — the package's importable/installed name.
  - `version='0.0.1'` — package version.
  - `author`, `author_email` — metadata only, no functional effect.
  - `packages=find_packages()` — the list of sub-packages to include (auto-detected).
  - `install_requires=get_requirements('requirements.txt')` — the dependency list, reusing `requirements.txt` as the single source of truth instead of duplicating package names in two places.

**How this connects back to `requirements.txt` and the earlier `pip install` step:**

`requirements.txt` ends with `-e .`. That line is what made `pip install -r requirements.txt` (run in Section 6) also install *this project itself* as an editable package — which is why the install log showed `Running setup.py develop for mlproject` and `Successfully installed ... mlproject`. "Editable" means the package is linked back to this source folder rather than copied into `site-packages`, so any code changes here are picked up immediately without reinstalling — useful during active development, and why `from src...` / `import mlproject...`-style imports work project-wide once the env is activated.